# 35 — Final Capstone: A Research-Quality Ultrasound Deep Learning Project

This is the final notebook of **PyTorch From First Principles**.

The objective is to connect everything into one trustworthy ultrasound research workflow.

We will combine:

- Research question definition
- Patient/study units
- Leakage-safe splitting
- Image and clinical preprocessing
- Baselines
- Transfer learning
- Multimodal fusion
- Study-level aggregation
- Domain shift and harmonization
- Cross-validation
- Ablations
- Calibration and uncertainty
- Explainability
- External validation
- Reproducible reporting


In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch:",torch.__version__)


# 1. The Capstone Principle

A research-quality model is not only an architecture.

It is the complete experimental system:

$$
\boxed{
Question+Data+Split+Preprocessing+Model+Validation+Robustness+Reporting
}
$$


# 2. Define the Clinical Prediction Task

Example:

> Given renal ultrasound images and pre-scan clinical features, predict one of three disease categories.

Define before modeling:

- Input
- Target
- Prediction time
- Prediction unit
- Primary metric
- Intended deployment setting


# 3. Define the Independent Unit

If the final decision is per patient:

$$
\boxed{Patient}
$$

should guide:

- Split
- CV
- Bootstrap
- Final reporting


# 4. Metadata Schema

A useful table may include:

$$
\begin{array}{|c|c|}
\hline
image\_path & Image\ file \\
\hline
patient\_id & Patient \\
\hline
study\_id & Examination \\
\hline
label & Target \\
\hline
site & Hospital \\
\hline
device & Scanner \\
\hline
view & Ultrasound\ view \\
\hline
age & Clinical\ variable \\
\hline
sex & Clinical\ variable \\
\hline
\end{array}
$$


# 5. Data Audit

Before modeling, check:

- Missing files
- Duplicate images
- Patient duplicates
- Multiple labels per patient
- Site/class association
- Device/class association
- Missing clinical data


In [ ]:
def audit_metadata(df):
    print("Rows:",len(df))
    print("Patients:",df["patient_id"].nunique())
    print("\nMissing:")
    print(df.isna().sum())
    print("\nClasses:")
    print(df["label"].value_counts())


# 6. Leakage Audit

Potential leakage:

- Patient overlap
- Neighboring frames across sets
- Test-derived normalization
- Test-derived harmonization
- Post-diagnosis clinical variables
- Duplicate images


# 7. Patient-Level Split


In [ ]:
def patient_split(patient_ids,seed=42,train_frac=.7,val_frac=.15):
    ids=list(patient_ids)
    random.Random(seed).shuffle(ids)
    n=len(ids);a=int(train_frac*n);b=int(val_frac*n)
    train=set(ids[:a]);val=set(ids[a:a+b]);test=set(ids[a+b:])
    assert train.isdisjoint(val)
    assert train.isdisjoint(test)
    assert val.isdisjoint(test)
    return train,val,test


# 8. Save Exact Split Manifest


In [ ]:
def save_split_manifest(path,train,val,test):
    payload={
        "train_patient_ids":sorted(train),
        "val_patient_ids":sorted(val),
        "test_patient_ids":sorted(test)
    }
    Path(path).write_text(json.dumps(payload,indent=2),encoding="utf-8")


# 9. Training-Only Preprocessing

Fit image:

$$
\mu_{train},\sigma_{train}
$$

and clinical:

- Medians
- Means/stds
- Category vocabularies

using training patients only.


# 10. Establish Baselines

Minimum useful baseline set:

1. Majority-class baseline
2. Clinical-only model
3. Small image CNN
4. Transfer-learning model


In [ ]:
def majority_accuracy(labels):
    y=torch.as_tensor(labels)
    counts=torch.bincount(y)
    return float(counts.max()/len(y))


# 11. Small CNN Baseline


In [ ]:
class CapstoneCNN(nn.Module):
    def __init__(self,num_classes=3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),nn.Linear(32,num_classes)
        )
    def forward(self,x): return self.net(x)


# 12. Transfer Learning

Compare pretrained and scratch models under identical:

- Patient folds
- Metrics
- Checkpoint rule
- Preprocessing policy


# 13. Multimodal Baselines

Compare:

$$
Image\ Only
$$

$$
Clinical\ Only
$$

$$
Image+Clinical
$$

using only features available at prediction time.


# 14. Study-Level Aggregation

If multiple frames exist:

- Mean probability
- Max probability
- Majority vote
- Attention MIL

Evaluate at the intended study/patient level.


In [ ]:
def mean_probability(probabilities):
    return probabilities.mean(dim=0)


# 15. Primary Metric

Choose before final testing.

Possible choices:

- Macro F1
- Sensitivity/specificity
- AUROC
- AUPRC
- Calibration


# 16. Patient-Grouped Cross-Validation

For small datasets, use grouped CV. Every fold refits preprocessing from fold-training patients only.


# 17. Multiple Seeds

Repeat key experiments using the same predefined seed set.

Do not report only the best seed.


# 18. Ablation Studies

Examples:

- No harmonization
- No clinical features
- No augmentation
- Scratch vs pretrained
- Mean pooling vs attention pooling


# 19. Fair Ablation Rule

$$
\boxed{
Same\ Folds+Same\ Seeds+Same\ Budget+One\ Controlled\ Change
}
$$


# 20. Domain Shift Audit

Inspect:

- Site × label
- Device × label
- Intensity by site
- Resolution by device
- Missingness by site


# 21. Harmonization

Compare:

- None
- Global normalization
- Per-image standardization
- Histogram matching
- Feature-space approaches

Fit learned harmonizers on training data only.


# 22. Harmonization Success

The correct question is:

$$
\boxed{
Does\ unseen\ site/device\ generalization\ improve?
}
$$


# 23. Site-Held-Out Evaluation

Train on familiar sites and evaluate on a site excluded from training.


# 24. Device-Held-Out Evaluation

Hold out a scanner family to test acquisition robustness.


# 25. Robustness Stress Tests

Evaluate sensitivity to:

- Gain
- Noise
- Blur
- Resolution
- Cropping


# 26. Calibration

Use validation predictions for:

- Reliability diagrams
- ECE
- Temperature scaling


In [ ]:
def confidence_from_logits(logits):
    return torch.softmax(logits,dim=1).max(dim=1).values


# 27. Uncertainty

Useful baselines:

- Seed/deep ensembles
- MC dropout
- Predictive entropy


# 28. Selective Prediction

Decide whether the model can abstain on uncertain cases.

Report coverage and risk.


# 29. Explainability

Use Grad-CAM/saliency/occlusion to investigate:

- Anatomy
- Borders
- Logos
- Markers
- Scanner artifacts

Use explanations as debugging evidence, not causal proof.


# 30. Error Analysis

Save per-patient:

- True label
- Prediction
- Probabilities
- Confidence
- Site
- Device
- View


# 31. High-Confidence Errors

These can reveal:

- Shortcut learning
- Label errors
- Domain shift


# 32. Patient-Level Bootstrap

Resample patients, not correlated frames.


In [ ]:
def bootstrap_accuracy(y_true,y_pred,n_bootstrap=1000,seed=42):
    y_true=torch.as_tensor(y_true)
    y_pred=torch.as_tensor(y_pred)
    n=len(y_true)
    g=torch.Generator().manual_seed(seed)
    values=[]
    for _ in range(n_bootstrap):
        idx=torch.randint(0,n,(n,),generator=g)
        values.append(float((y_true[idx]==y_pred[idx]).float().mean()))
    values=torch.tensor(values)
    return {
        "estimate":float((y_true==y_pred).float().mean()),
        "lower":float(torch.quantile(values,.025)),
        "upper":float(torch.quantile(values,.975))
    }


# 33. External Validation

The external cohort should not influence:

- Architecture
- Harmonization
- Hyperparameters
- Threshold
- Ensemble composition


# 34. External Evaluation

Report:

- Overall patient-level metric
- Class-wise metrics
- Site/device robustness
- Calibration
- Confidence intervals


# 35. Save Raw Predictions

Prediction-level files enable later recalculation of metrics, calibration, bootstrap intervals, and failure analyses.


# 36. Reproducible Experiment Folder


In [ ]:
CAPSTONE_DIR=Path("capstone_experiment")
CAPSTONE_DIR.mkdir(parents=True,exist_ok=True)

config={
    "project":"research_quality_ultrasound_capstone",
    "split_unit":"patient",
    "primary_metric":"macro_f1",
    "seeds":[11,22,33],
    "external_validation":True
}

(CAPSTONE_DIR/"config.json").write_text(
    json.dumps(config,indent=2),
    encoding="utf-8"
)

print(CAPSTONE_DIR)


# 37. Final Result Tables

Create programmatically:

- Main model table
- Ablation table
- Robustness table
- External-validation table
- Calibration table


# 38. Methods Section

Document:

- Cohort
- Labels
- Inclusion/exclusion
- Split design
- Preprocessing
- Models
- Optimization
- Metrics
- Statistics
- External validation


# 39. Results Section

Report:

- Patient counts
- Class counts
- Internal performance
- External performance
- Confidence intervals
- Ablations
- Robustness


# 40. Discussion

Discuss:

- What worked
- What failed
- Generalization
- Bias
- Limitations
- Clinical meaning
- Future work


# 41. Negative Results Matter

If harmonization does not help external generalization, report it. Negative results are scientifically informative.


# 42. Statistical vs Clinical Significance

A statistically detectable improvement may still be too small to matter clinically.


# 43. Data Quantity vs Complexity

With limited patients, a simpler model may generalize better than a much larger architecture.


# 44. More Independent Patients Beat More Lucky Seeds

Multiple seeds estimate optimization variability. They do not create new biological diversity.


# 45. Common Mistakes

- Patient leakage
- Validation overfitting
- Test-set tuning
- Ignoring site/device shift
- Multimodal leakage
- Harmonization leakage
- Reporting only one metric
- Reporting only best fold/seed
- Treating explanations as validation
- Failing to save predictions


# 46. Final Capstone Checklist

1. Task defined
2. Prediction time defined
3. Patient split verified
4. Leakage audit complete
5. Baselines trained
6. Transfer model compared
7. Multimodal ablation complete
8. Harmonization compared
9. Grouped CV complete
10. Multiple seeds run
11. External test untouched
12. Calibration evaluated
13. Uncertainty evaluated
14. Explainability sanity checks complete
15. Confidence intervals reported
16. Configs/predictions/checkpoints saved


# 47. What You Can Build Now

After 35 notebooks you can build projects involving:

- Custom datasets
- CNNs
- Transfer learning
- Vision Transformers
- Segmentation
- Self-supervised learning
- Multimodal fusion
- Study-level MIL
- Domain shift
- Harmonization
- Calibration
- Deployment
- Reproducible research engineering


# 48. Final Research Workflow

$$
\boxed{
Raw\ Data
\rightarrow
Audit
\rightarrow
Patient\ Splits
\rightarrow
Training\ Preprocessing
\rightarrow
Baselines
\rightarrow
Advanced\ Models
\rightarrow
CV/Ablations
\rightarrow
Robustness
\rightarrow
External\ Validation
\rightarrow
Report
}
$$


# 49. Final Exercises

1. Write your exact ultrasound prediction task.
2. Design your metadata schema.
3. Define your patient split protocol.
4. Define image-only/clinical-only/fusion baselines.
5. Design one harmonization ablation.
6. Design one site-held-out experiment.
7. Define calibration analysis.
8. Define patient bootstrap.
9. Design experiment folder structure.
10. Draft your methods-section outline.


# 50. Final Takeaways

The deepest question is not:

> Which model is newest?

It is:

> **How do I build an experiment whose result I can trust?**

Protect:

$$
\boxed{Patient\ Independence}
$$

use:

$$
\boxed{Training-Only\ Preprocessing}
$$

compare:

$$
\boxed{Strong\ Baselines}
$$

test:

$$
\boxed{Unseen\ Domains}
$$

and report:

$$
\boxed{Uncertainty+Failures+Reproducible\ Evidence}
$$


# Course Complete

# PyTorch From First Principles — 35 Notebooks Complete

You have progressed from tensors and autograd to research-grade ultrasound deep learning.

The natural next step is to apply the complete workflow to your real dataset and turn it into a full research project.
